# 00 · Data Exploration: Tirosh et al. 2016 (GSE72056)

**Goal.** Before any analysis, establish exactly what the public data looks like:

1. Confirm the file can be loaded in Python / Jupyter.
2. Determine the matrix orientation (what are rows vs. columns?).
3. Determine the value type — log-normalized expression or raw counts? *This decides which downstream methods are valid.*
4. Verify the reported metadata structure (the paper states the first rows encode cell-level metadata).

**Dataset.** GEO accession `GSE72056`, supplementary file
`GSE72056_melanoma_single_cell_revised_v2.txt.gz` — Tirosh et al.,
*Science* 2016, "Dissecting the multicellular ecosystem of metastatic
melanoma by single-cell RNA-seq" (SMART-seq2, 4,645 single cells).

## Step 0 — Imports and locate the data file

- `pandas` (`pd`): tabular data handling.
- `numpy` (`np`): numerical operations on the value matrix.
- `pathlib.Path`: robust path handling across run locations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve the path whether the notebook runs from notebooks/ or the repo root.
FNAME = "GSE72056_melanoma_single_cell_revised_v2.txt.gz"
RAW = Path("../data/raw") / FNAME
if not RAW.exists():
    RAW = Path("data/raw") / FNAME

assert RAW.exists(), f"Data file not found. Download it into data/raw/ first: {RAW}"
print("Data file:", RAW.resolve())
print("Compressed size: %.1f MB" % (RAW.stat().st_size / 1e6))

Data file: /Users/youyouwu/Desktop/melanoma-scrnaseq-reanalysis/data/raw/GSE72056_melanoma_single_cell_revised_v2.txt.gz
Compressed size: 75.0 MB


## Step 1 — Load the matrix and inspect its shape

The file is tab-separated text (`.txt`), gzip-compressed (`.gz`).
`pandas` decompresses `.gz` transparently from the file extension.

- `sep="\t"`: tab is the column delimiter.
- `index_col=0`: use the first column (gene / metadata labels) as the row
  index, so the remaining columns are exactly the single cells.

Loading takes ~10–15 s (the decompressed matrix is several hundred MB).

In [2]:
df = pd.read_csv(RAW, sep="\t", index_col=0)

print("df.shape =", df.shape, " (rows, columns)")
print("Row index (labels), first 5:", list(df.index[:5]))
print("Columns (cell IDs), first 3:", list(df.columns[:3]))
# Every column mixes integer metadata codes and gene expression values,
# all numeric -> pandas infers a single numeric dtype.
print("Column dtype(s):", df.dtypes.unique())

df.shape = (23689, 4645)  (rows, columns)
Row index (labels), first 5: ['tumor', 'malignant(1=no,2=yes,0=unresolved)', 'non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)', 'C9orf152', 'RPS11']
Columns (cell IDs), first 3: ['Cy72_CD45_H02_S758_comb', 'CY58_1_CD45_B02_S974_comb', 'Cy71_CD45_D08_S524_comb']
Column dtype(s): [dtype('float64')]


## Step 2 — Inspect the first rows and columns

`.iloc[:6, :4]` selects the first 6 rows and 4 columns (`iloc` indexes by
integer position, starting at 0). Note that the first three **row labels**
are not gene names — they describe each cell.

In [3]:
df.iloc[:6, :4]

,Cy72_CD45_H02_S758_comb,CY58_1_CD45_B02_S974_comb,Cy71_CD45_D08_S524_comb,Cy81_FNA_CD45_B01_S301_comb
Cell,,,,
tumor,72.0000,58.0000,71.0000,81.0000
"malignant(1=no,2=yes,0=unresolved)",1.0000,1.0000,2.0000,2.0000
"non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)",2.0000,1.0000,0.0000,0.0000
C9orf152,0.0000,0.0000,0.0000,0.0000
RPS11,9.2172,8.3745,9.3130,7.8876
ELMO2,0.0000,0.0000,2.1263,0.0000


## Step 3 — Separate cell metadata from the expression matrix

The first three rows are **per-cell metadata**, not genes:

| Row label | Meaning |
|---|---|
| `tumor` | Patient / tumor sample of origin (numeric ID) |
| `malignant(1=no,2=yes,0=unresolved)` | Malignancy call for the cell |
| `non-malignant cell type (1=T,2=B,3=Macro,4=Endo,5=CAF,6=NK)` | Cell-type label for non-malignant cells (0 for malignant cells) |

Gene expression begins at row 4. We split the two parts.

In [4]:
meta = df.iloc[:3]   # first 3 rows  -> per-cell metadata
expr = df.iloc[3:]   # remaining rows -> gene expression matrix

print("Metadata row labels:")
for name in meta.index:
    print("  -", name)
print()
print("Expression matrix:", expr.shape, "->", expr.shape[0], "genes x", expr.shape[1], "cells")
print()
# Transpose so rows are cells and columns are the 3 metadata fields.
print("Metadata for the first 5 cells:")
meta.T.head()

Metadata row labels:
  - tumor
  - malignant(1=no,2=yes,0=unresolved)
  - non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)

Expression matrix: (23686, 4645) -> 23686 genes x 4645 cells

Metadata for the first 5 cells:


Cell,tumor,"malignant(1=no,2=yes,0=unresolved)","non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)"
Cy72_CD45_H02_S758_comb,72.0,1.0,2.0
CY58_1_CD45_B02_S974_comb,58.0,1.0,1.0
Cy71_CD45_D08_S524_comb,71.0,2.0,0.0
Cy81_FNA_CD45_B01_S301_comb,81.0,2.0,0.0
Cy80_II_CD45_B07_S883_comb,80.0,2.0,0.0


In [5]:
# Sanity check: how many malignant vs. non-malignant cells?
mal = meta.loc["malignant(1=no,2=yes,0=unresolved)"]
legend = {1.0: "non-malignant", 2.0: "malignant", 0.0: "unresolved"}
print(mal.map(legend).value_counts())

malignant(1=no,2=yes,0=unresolved)
non-malignant    3256
malignant        1257
unresolved        132
Name: count, dtype: int64


## Step 4 — Determine the value type: log-normalized or raw counts?

Decision rule:

- **Raw counts**: non-negative integers; maximum typically in the
  hundreds–thousands.
- **Log-normalized** (e.g. `log2(TPM/10+1)`): continuous decimals,
  typically in the `0–15` range.

**Pitfall.** The naive "fraction of integer values" is misleading: scRNA-seq
matrices are very sparse (~80% of entries are `0`, and `0` is an integer).
The correct test is the integer fraction **among non-zero values only**.

In [6]:
# float32 halves memory vs. float64 for this large matrix.
a = expr.to_numpy(dtype=np.float32)

print("min  = %.4f" % a.min())
print("max  = %.4f" % a.max())
print("mean = %.4f" % a.mean())
print("quantiles 50%% / 90%% / 99%% = %.3f / %.3f / %.3f"
      % tuple(np.quantile(a, [0.5, 0.9, 0.99])))
print()

zero_frac = np.mean(a == 0)
print("fraction of zeros            = %.1f%%  (matrix is sparse)" % (100 * zero_frac))

nz = a[a > 0]                                   # non-zero entries only
int_frac_all = np.mean(a == np.round(a))
int_frac_nz  = np.mean(nz == np.round(nz))
print("integer fraction (incl. 0)   = %.1f%%  (inflated only by the zeros)" % (100 * int_frac_all))
print("integer fraction (non-zero)  = %.2f%%  (key metric: ~0 -> continuous)" % (100 * int_frac_nz))
print()
print("Example non-zero values:", np.round(nz[:8], 4).tolist())

min  = 0.0000
max  = 15.9230
mean = 0.4644


quantiles 50% / 90% / 99% = 0.000 / 1.785 / 6.200



fraction of zeros            = 81.3%  (matrix is sparse)


integer fraction (incl. 0)   = 81.3%  (inflated only by the zeros)
integer fraction (non-zero)  = 0.03%  (key metric: ~0 -> continuous)

Example non-zero values: [0.12430000305175781, 1.0454000234603882, 0.3752000033855438, 0.13220000267028809, 0.501800000667572, 0.9567999839782715, 3.4758999347686768, 0.8984000086784363]


## Conclusions

| Check | Result |
|---|---|
| Loads in Jupyter | Yes (`pd.read_csv` reads the `.gz` directly) |
| Overall shape | **23,689 rows x 4,645 columns** |
| Rows | **Genes** (plus 3 metadata rows at the top) |
| Columns | **Cells** (4,645 single cells) |
| First 3 rows are metadata | Confirmed: `tumor` / `malignant` / `non-malignant cell type` |
| Number of genes | **23,686** (23,689 − 3 metadata rows) |
| Value range | min = 0, max ≈ **15.9**, median = 0 |
| Value type | **`log2(TPM/10 + 1)`** — not raw counts |

**Evidence.** Among non-zero entries only ~0.03% are integers (values are
continuous), and the maximum (~15.9) is the expected magnitude in log space;
raw counts would be non-negative integers with a much larger maximum. This
matches the transformation `Ei,j = log2(TPM/10 + 1)` described in Tirosh
et al. 2016. The platform is SMART-seq2 (plate-based, full-length, TPM —
**no UMIs**).

**Implications for downstream methods.**

- The data is **already in log space and normalized** — do not re-apply
  `log1p` or library-size normalization.
- Count-based models (scVI / scANVI negative-binomial likelihoods,
  `highly_variable_genes(flavor="seurat_v3")`, Poisson/NB differential
  expression) assume raw counts and **cannot consume this matrix directly**;
  using them requires either obtaining raw counts from the authors or an
  explicit, documented count-approximation step.
- Methods valid on log-normalized input (PCA → kNN graph → Leiden → UMAP,
  Wilcoxon/t-test marker genes, `flavor="seurat"`/`"cell_ranger"` HVGs) can
  be used directly — this is essentially the original paper's approach.
- When building an `AnnData` object, **transpose** to cells × genes
  (scanpy's convention), the opposite of this file's layout.

➡️ Next notebook: construct the `AnnData` object and run quality control.